# Building Age Classification from Satellite Imagery

# Notebook 6-Model Zoo: which Stage 1 model transfers best?

---

Notebook 5 built our transfer-oriented features and a two-stage model with **LightGBM** as Stage 1. Here we keep **everything else fixed** (features, Stage 2 methods, evaluation draws) and swap the Stage 1 model. This isolates one question: *which model family learns the most transferable knowledge from Madrid?*

| Candidate | Family |
|---|---|
| LightGBM (Notebook 5) | Gradient boosting, leaf-wise trees |
| XGBoost | Gradient boosting, depth-wise trees |
| CatBoost | Gradient boosting, symmetric (oblivious) trees |
| HistGradientBoosting | scikit-learn gradient boosting |
| Random Forest | Bagged deep trees (the baseline family) |
| Extra Trees | Bagged randomised trees |
| MLP | Neural network on the same features |
| **Ensemble** | Average of all the models' log-probabilities |

### Prerequisite: run Notebook 5 first

This notebook re-uses Notebook 5's outputs instead of recomputing them:

* `data/cache/features_{city}.parquet`: the 166 engineered features (city-year anomaly, change-point, neighbourhood context)
* `data/cache/cube_{city}.npz`: labels (`weighted_mean_year`) and pixel coordinates
* `models/stage1_madrid_year_regressor.txt`: the Madrid construction-year model (an input to `calibrate`)

### What each stage is allowed to see

| Step | Madrid labels | Amsterdam labels | Unlabelled Amsterdam pixels |
|---|:-:|:-:|:-:|
| Feature engineering + per-city scaling | — | ✗ | ✓ (city-year medians/IQRs; disclosed on our slides) |
| Stage 1 training | ✓ | ✗ | ✗ |
| Stage 2 adaptation | — | only the *n*-per-class support set | ✗ |
| Scoring | — | only pixels **not** in the support set | — |

## 6.1 Setup

In [ ]:
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightgbm as lgb
import xgboost as xgb                       # if missing on your platform:  !pip install xgboost catboost
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import f1_score, confusion_matrix
from sklearn.utils.class_weight import compute_sample_weight

warnings.filterwarnings('ignore', category=RuntimeWarning)
warnings.filterwarnings('ignore', message='X does not have valid feature names')
plt.rcParams['figure.dpi'] = 100

# ── Config ────────────────────────────────────────────────────────────────────
SEED       = 42
SHOTS      = [5, 25, 50, 100, 200]   # labelled Amsterdam pixels per class
N_TRIALS   = 10                      # random support draws per shot size (same draws for every model)
ZOO_FOLDS  = 3                       # spatial-block CV folds on Madrid per model
N_JOBS     = 4                       # CPU threads
EPS        = 1e-3                    # probability floor before logs (RF/ET can output exact zeros)
QUICK_TEST = False               # True = tiny subsample, just to check the notebook runs

DATA_DIR  = next(p for p in [Path('data'), Path('../data')] if (p / 'madrid_train.parquet').exists())
CACHE_DIR = DATA_DIR / 'cache'
MODEL_DIR = Path('models')
ZOO_CACHE = CACHE_DIR / 'zoo'; ZOO_CACHE.mkdir(parents=True, exist_ok=True)

CLASS_EVENTS = {'Amsterdam': 1945, 'Madrid': 1960}
BASELINE = {'Madrid CV': 0.618, 'zero-shot': 0.343, 5: 0.544, 25: 0.599, 50: 0.608, 100: 0.613, 200: 0.615}
NB5_LGBM = {5: 0.557, 25: 0.650, 50: 0.680, 100: 0.714, 200: 0.731}   # Notebook 5 final results

## 6.2 Load Notebook 5's features and labels

We apply the same per-city robust scaling as Notebook 5: each feature is scaled by its own city's median and IQR, using unlabelled pixels only.

In [ ]:
need = [CACHE_DIR / f'{k}_{c}.{e}' for c in ('Madrid', 'Amsterdam') for k, e in (('features', 'parquet'), ('cube', 'npz'))]
need.append(MODEL_DIR / 'stage1_madrid_year_regressor.txt')
missing = [str(p) for p in need if not p.exists()]
assert not missing, f'Run Notebook 5 first. Missing: {missing}'


def age_class(year, city):
    bins = [-np.inf, CLASS_EVENTS[city], 1984, 2004, np.inf]
    return pd.cut(year, bins=bins, labels=[1, 2, 3, 4]).astype(int)


def city_scale(df):
    # Robust z-score of every feature against its own city (unsupervised; no labels used).
    return (df - df.median()) / (df.quantile(.75) - df.quantile(.25) + 1e-9)


META, X, Y = {}, {}, {}
for city in ('Madrid', 'Amsterdam'):
    z = np.load(CACHE_DIR / f'cube_{city}.npz')
    META[city] = {k: z[k] for k in ('px', 'py', 'wmy')}
    Y[city] = age_class(META[city]['wmy'], city)
    feat = pd.read_parquet(CACHE_DIR / f'features_{city}.parquet')
    X[city] = city_scale(feat).values.astype(np.float32)
FEATURE_NAMES = list(feat.columns)

Xm, ym = X['Madrid'], Y['Madrid']
Xa, ya = X['Amsterdam'], Y['Amsterdam']
groups_m = None

if QUICK_TEST:   # small random subsample + fewer draws, only to smoke-test the notebook
    idx = np.random.default_rng(0).choice(len(ym), 4000, replace=False)
    Xm, ym = Xm[idx], ym[idx]
    META['Madrid'] = {k: v[idx] for k, v in META['Madrid'].items()}
    SHOTS, N_TRIALS = [5, 25], 2

# Madrid year regressor from Notebook 5 (Stage 1b): predicted construction year for Amsterdam
ams_year = lgb.Booster(model_file=str(MODEL_DIR / 'stage1_madrid_year_regressor.txt')).predict(Xa)

print(f'Madrid {Xm.shape}  classes {np.bincount(ym)[1:].tolist()}')
print(f'Amsterdam {Xa.shape}  classes {np.bincount(ya)[1:].tolist()}')

## 6.3 Shared helpers (same as Notebook 5)

* `spatial_blocks`: 1 km × 1 km block ids. Whole blocks are held out in CV, so neighbouring pixels never sit on both sides of a split (guidance point 3).
* `finetune_booster`: Notebook 5's Stage 2. It continues boosting from a Stage 1 model's scores with small LightGBM trees. Because it only needs the Stage 1 **log-probabilities** as a starting point (`init_score`), it works on top of **any** model.
* `plot_cm`: row-normalised confusion matrix (per-class recall).

In [ ]:
def spatial_blocks(px, py, block_m=1000):
    cells = int(round(block_m / 30))
    return ((px - px.min()) // cells).astype(np.int64) * 100_000 + (py - py.min()) // cells


def finetune_booster(X_sup, y_sup, init_scores, n, seed=SEED):
    params = dict(objective='multiclass', num_class=4, learning_rate=0.05, num_leaves=7,
                  min_data_in_leaf=max(3, n // 5), feature_fraction=0.5, lambda_l2=5.0,
                  num_threads=N_JOBS, seed=seed, verbose=-1)
    ds = lgb.Dataset(X_sup, y_sup - 1, init_score=init_scores)
    return lgb.train(params, ds, num_boost_round=int(np.clip(n, 10, 150)))


def plot_cm(ax, y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred, labels=[1, 2, 3, 4]).astype(float)
    cm /= cm.sum(1, keepdims=True)
    ax.imshow(cm, cmap='Blues', vmin=0, vmax=1)
    for i in range(4):
        for j in range(4):
            ax.text(j, i, f'{cm[i, j]:.2f}', ha='center', va='center', color='white' if cm[i, j] > .6 else 'black')
    ax.set_xticks(range(4)); ax.set_xticklabels([f'C{k}' for k in range(1, 5)])
    ax.set_yticks(range(4)); ax.set_yticklabels([f'C{k}' for k in range(1, 5)])
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual'); ax.set_title(title, fontweight='bold', fontsize=10)


groups_m = spatial_blocks(META['Madrid']['px'], META['Madrid']['py'])
print(f'{len(np.unique(groups_m)):,} spatial blocks of 1 km in Madrid')

## 6.4 The candidate models

All models get **balanced class weights**, because class 2 dominates Madrid. Hyper-parameters are sensible defaults rather than tuned, so the comparison is about the model *family*.

In [ ]:
class XGBBalanced(xgb.XGBClassifier):
    # XGBoost with balanced sample weights; accepts/returns labels 1-4 (XGBoost itself uses 0-3).
    def fit(self, X, y):
        super().fit(X, y - 1, sample_weight=compute_sample_weight('balanced', y))
        return self
    def predict(self, X):
        return super().predict(X) + 1


def clip_extremes(Z):
    # Neural nets are sensitive to extreme inputs; robust z-scores can reach +-50.
    return np.clip(Z, -8, 8)


STAGE1_PARAMS = dict(n_estimators=400, learning_rate=0.05, num_leaves=63, subsample=0.8, subsample_freq=1,
                     colsample_bytree=0.5, class_weight='balanced', n_jobs=N_JOBS, verbose=-1, random_state=SEED)

ZOO = {
    'LightGBM':      lambda: lgb.LGBMClassifier(**STAGE1_PARAMS),
    'XGBoost':       lambda: XGBBalanced(n_estimators=400, learning_rate=0.05, max_depth=7, subsample=0.8,
                                         colsample_bytree=0.5, tree_method='hist', n_jobs=N_JOBS,
                                         random_state=SEED, verbosity=0),
    'CatBoost':      lambda: CatBoostClassifier(iterations=600, depth=6, learning_rate=0.08,
                                                auto_class_weights='Balanced', thread_count=N_JOBS,
                                                random_seed=SEED, verbose=0, allow_writing_files=False),
    'HistGradBoost': lambda: HistGradientBoostingClassifier(max_iter=300, learning_rate=0.08, max_leaf_nodes=63,
                                                            class_weight='balanced', random_state=SEED),
    'RandomForest':  lambda: RandomForestClassifier(n_estimators=300, min_samples_leaf=2, max_features='sqrt',
                                                    class_weight='balanced', n_jobs=N_JOBS, random_state=SEED),
    'ExtraTrees':    lambda: ExtraTreesClassifier(n_estimators=300, min_samples_leaf=2, max_features='sqrt',
                                                  class_weight='balanced', n_jobs=N_JOBS, random_state=SEED),
    'MLP':           lambda: make_pipeline(FunctionTransformer(clip_extremes),
                                           MLPClassifier(hidden_layer_sizes=(256, 128), alpha=1e-3, batch_size=512,
                                                         max_iter=60, early_stopping=True, random_state=SEED)),
}
RUN_MODELS = list(ZOO)  
print('Models:', RUN_MODELS)

## 6.5 Stage 1: Madrid spatial CV + zero-shot on Amsterdam

For each model:
1. **Spatial-block CV on Madrid** (`ZOO_FOLDS` folds, the same folds for every model).
2. **Final fit on all of Madrid**, then predict Amsterdam class probabilities. That gives the zero-shot score and the starting point for Stage 2.

Results are cached in `data/cache/zoo/`, so re-running skips models that are already done. Delete that folder to start fresh.
On a 4-core laptop the full zoo takes roughly 30–60 minutes. Random Forest, Extra Trees and CatBoost are the slowest.

In [ ]:
def predict_labels(model, X):
    return np.asarray(model.predict(X)).ravel().astype(int)     # CatBoost returns a column vector


zoo_cv, zoo_logp = {}, {}
cv = StratifiedGroupKFold(ZOO_FOLDS, shuffle=True, random_state=SEED)
folds = list(cv.split(Xm, ym, groups_m))
tag = '_quick' if QUICK_TEST else ''

for name in RUN_MODELS:
    path = ZOO_CACHE / f'{name}{tag}.npz'
    if path.exists():
        z = np.load(path); zoo_cv[name] = z['cv']; zoo_logp[name] = z['logp']
        print(f'{name:14s} (cached)', end='  ')
    else:
        t0 = time.time()
        f1s = []
        for tr, va in folds:
            m = ZOO[name]().fit(Xm[tr], ym[tr])
            f1s.append(f1_score(ym[va], predict_labels(m, Xm[va]), average='macro'))
        final = ZOO[name]().fit(Xm, ym)
        zoo_cv[name] = np.array(f1s)
        zoo_logp[name] = np.log(np.clip(final.predict_proba(Xa), EPS, 1)).astype(np.float32)
        np.savez(path, cv=zoo_cv[name], logp=zoo_logp[name])
        print(f'{name:14s} {time.time() - t0:5.0f}s', end='  ')
    zero = f1_score(ya, zoo_logp[name].argmax(1) + 1, average='macro')
    print(f'Madrid spatial CV {zoo_cv[name].mean():.4f} ± {zoo_cv[name].std():.4f} | Amsterdam zero-shot {zero:.4f}')

# Ensemble: average log-probabilities of all models (a geometric mean of their probabilities)
zoo_logp['Ensemble'] = np.mean([zoo_logp[m] for m in RUN_MODELS], axis=0)
zoo_cv['Ensemble'] = np.array([np.nan])
print(f"{'Ensemble':14s}                                   | Amsterdam zero-shot "
      f"{f1_score(ya, zoo_logp['Ensemble'].argmax(1) + 1, average='macro'):.4f}")

## 6.6 Stage 2: few-shot adaptation for every model

Two model-agnostic Stage 2 methods, run on **identical support draws** for every model (same seed as Notebook 5):

* **`calibrate`**: logistic regression on [model log-probabilities, predicted year]. A light classifier on top of a *frozen* Stage 1.
* **`finetune`**: small LightGBM correction trees boosted on top of the model's log-probabilities. This adapts both the outputs and the features.

Every score is computed on the Amsterdam pixels **not** in the support set.

In [ ]:
def stage2_calibrate(logp, sup, n):
    Z = StandardScaler().fit_transform(np.column_stack([logp, ams_year]))
    C = 0.05 if n <= 5 else 0.3 if n <= 50 else 1.0          # more regularisation with fewer labels
    lr = LogisticRegression(C=C, max_iter=2000, class_weight='balanced').fit(Z[sup], ya[sup])
    return lr.predict(Z)


def stage2_finetune(logp, sup, n):
    booster = finetune_booster(Xa[sup], ya[sup], logp[sup], n)
    return (logp + booster.predict(Xa, raw_score=True)).argmax(1) + 1


STAGE2 = {'calibrate': stage2_calibrate, 'finetune': stage2_finetune}
zoo_res = {(m, s): {n: [] for n in SHOTS} for m in zoo_logp for s in STAGE2}
rng = np.random.default_rng(SEED)
t0 = time.time()
for n in SHOTS:
    for trial in range(N_TRIALS):
        sup = np.concatenate([rng.choice(np.where(ya == c)[0], n, replace=False) for c in range(1, 5)])
        query = np.ones(len(ya), bool); query[sup] = False
        for m, logp in zoo_logp.items():
            for s, fn in STAGE2.items():
                pred = fn(logp, sup, n)
                zoo_res[(m, s)][n].append(f1_score(ya[query], pred[query], average='macro'))
    print(f'n = {n:3d}: ' + '  '.join(f"{m} {np.mean(zoo_res[(m, 'finetune')][n]):.3f}" for m in zoo_logp))
print(f'\nStage 2 done in {time.time() - t0:.0f}s')

## 6.7 Results

Each cell is the mean macro-F1 over `N_TRIALS` draws. Green marks the best value per column. The `avg F1` column is the "best average" prize metric, and `n=25` is the "low-data" prize metric.

In [ ]:
rows = []
for (m, s), r in zoo_res.items():
    means = [np.mean(r[n]) for n in SHOTS]
    rows.append({'Stage 1': m, 'Stage 2': s,
                 'Madrid spatial CV': np.nanmean(zoo_cv[m]),
                 'zero-shot': f1_score(ya, zoo_logp[m].argmax(1) + 1, average='macro'),
                 **{f'n={n}': v for n, v in zip(SHOTS, means)},
                 'avg F1': np.mean(means)})
zoo_table = pd.DataFrame(rows).sort_values('avg F1', ascending=False).reset_index(drop=True)
metric_cols = ['Madrid spatial CV', 'zero-shot'] + [f'n={n}' for n in SHOTS] + ['avg F1']
display(zoo_table.style.format(precision=3, na_rep='—').highlight_max(subset=metric_cols, color='#c7e9c0'))

best = zoo_table.iloc[0]
print(f"Best: {best['Stage 1']} + {best['Stage 2']}   avg F1 {best['avg F1']:.3f}")
print(f"Notebook 4 baseline avg F1 {np.mean([BASELINE[n] for n in SHOTS if n in BASELINE]):.3f} | "
      f"Notebook 5 (LightGBM + finetune) avg F1 {np.mean([NB5_LGBM[n] for n in SHOTS]):.3f}")

In [ ]:
# Mean ± std table for the finetune variant (the format the slides need)
ft = pd.DataFrame({m: [f"{np.mean(zoo_res[(m, 'finetune')][n]):.3f} ± {np.std(zoo_res[(m, 'finetune')][n]):.3f}"
                       for n in SHOTS] for m in zoo_logp}, index=[f'n = {n}' for n in SHOTS]).T
ft.loc['Notebook 4 baseline'] = [f'{BASELINE[n]:.3f}' for n in SHOTS]
display(ft)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6), sharey=True)
x = np.log2(SHOTS)
for ax, s in zip(axes, STAGE2):
    for m in zoo_logp:
        r = zoo_res[(m, s)]
        ax.errorbar(x, [np.mean(r[n]) for n in SHOTS], yerr=[np.std(r[n]) for n in SHOTS], marker='o', capsize=3,
                    lw=2.8 if m == 'Ensemble' else 1.3, ls='-' if m != 'Ensemble' else '--', label=m)
    ax.plot(x, [BASELINE[n] for n in SHOTS], 'k:', lw=1.5, label='Notebook 4 baseline')
    ax.set_xticks(x); ax.set_xticklabels([f'{n}\n(log2={np.log2(n):.1f})' for n in SHOTS])
    ax.set_xlabel('Labelled Amsterdam pixels per class'); ax.set_title(f'Stage 2 = {s}', fontweight='bold')
    ax.grid(alpha=.3)
axes[0].set_ylabel('Macro F1 (held-out Amsterdam pixels)'); axes[1].legend(fontsize=9, loc='lower right')
plt.suptitle('Model zoo: Amsterdam F1 vs labels per class (mean ± std over draws)', fontweight='bold')
plt.tight_layout(); plt.show()

## 6.8 Per-class behaviour of the best combination

Classes 3 and 4 were built during the satellite record, so their construction event can be visible. Classes 1 and 2 predate it. These are different problems, so we check where each model's errors fall.

In [ ]:
best_m, best_s = best['Stage 1'], best['Stage 2']
rng_cm = np.random.default_rng(SEED)
show_n = [n for n in (5, 25, 200) if n in SHOTS]
fig, axes = plt.subplots(1, len(show_n) + 1, figsize=(5 * (len(show_n) + 1), 4.5))
plot_cm(axes[0], ya, zoo_logp[best_m].argmax(1) + 1, f'{best_m}: zero-shot')
for ax, n in zip(axes[1:], show_n):
    sup = np.concatenate([rng_cm.choice(np.where(ya == c)[0], n, replace=False) for c in range(1, 5)])
    query = np.ones(len(ya), bool); query[sup] = False
    pred = STAGE2[best_s](zoo_logp[best_m], sup, n)
    plot_cm(ax, ya[query], pred[query], f'{best_m} + {best_s}, n={n}\nF1 = {f1_score(ya[query], pred[query], average="macro"):.3f}')
plt.tight_layout(); plt.show()

# Per-class F1 at n = 25 for every model (finetune)
rng_pc = np.random.default_rng(SEED)
n = 25 if 25 in SHOTS else SHOTS[0]
sup = np.concatenate([rng_pc.choice(np.where(ya == c)[0], n, replace=False) for c in range(1, 5)])
query = np.ones(len(ya), bool); query[sup] = False
per_class = pd.DataFrame({m: f1_score(ya[query], stage2_finetune(zoo_logp[m], sup, n)[query], average=None)
                          for m in zoo_logp}, index=[f'Class {k}' for k in range(1, 5)]).T
display(per_class.style.format(precision=3).background_gradient(cmap='Greens', axis=None))